In [ ]:
"""
UNICEF Intervention Simulator - PROOF OF IMPACT VERSION
Updated with two-track strategy focusing on high-alignment donors
"""

import pandas as pd
import asyncio
import aiohttp
import json
import os
import time
from typing import Dict

try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nest_asyncio", "--break-system-packages"])
    import nest_asyncio
    nest_asyncio.apply()

CONFIG = {
    "groq_api_key": os.environ.get("GROQ_API_KEY", ""),  # set GROQ_API_KEY in your environment (see .env.example)
    "groq_model": "qwen/qwen3-32b",
    "temperature": 0.0,
    "max_tokens": 800,
    "timeout_seconds": 45,
    "max_retries": 3,
    "concurrent_limit": 20,
    "input_file": "fragile_donors_with_prompts.csv",
    "output_file": "simulated_intervention_results_proof_of_impact.csv"
}

# NEW INTERVENTION: Proof of Impact (for high-alignment donors)
INTERVENTION_TEMPLATE = """UNICEF confirms your priorities align {alignment_pct:.0f}% with their focus. Because you're already well-aligned, they show you PROOF of your €{tot25_medium:.0f}'s impact:

• €{amount_education:.2f} funded 0.3 teachers' salaries + school supplies for 4 children 
  (Photo: classroom with "Funded by donors like you" sign)
  
• €{amount_health:.2f} vaccinated 8 children against measles 
  (Evidence: Vaccination records from clinic, your donation ID visible)
  
• €{amount_water:.2f} maintained 1 water well for 3 months 
  (Evidence: GPS coordinates 1°17'21"S 36°49'02"E + photo of well with date stamp)

• €{amount_overhead:.2f} overhead (field staff, logistics, monitoring)

You receive a dashboard link to track these specific programs in real-time with monthly updates showing concrete progress.

This is tangible proof with photos, GPS coordinates, and official records - not just generic reports."""

SYSTEM_PROMPT = """You are a sophisticated simulation engine for computational social science research. Your task is to embody a specific donor profile and respond authentically to an intervention showing concrete proof of impact.

CRITICAL INSTRUCTIONS:
1. Stay completely in character based on the donor backstory provided
2. This donor has HIGH ALIGNMENT (94% average) - they already care about UNICEF's work
3. The problem is LOW TRUST and LOW PERCEIVED IMPACT - not alignment
4. The intervention provides CONCRETE PROOF: photos, GPS coordinates, vaccination records
5. Most donors should respond POSITIVELY to concrete evidence (trust and impact should increase)
6. However, maintain realistic variation - some donors may be skeptical or unchanged
7. Output ONLY valid JSON - no preamble, no markdown formatting, no explanation outside the JSON

RESPONSE GUIDELINES:
- new_trust: Should typically INCREASE when shown concrete proof (0.0 → 0.5 or 1.0)
- new_impact: Should typically INCREASE when shown measurable results (0.0 → 0.5 or 1.0)
- new_alignment_pct: Should STAY THE SAME or increase slightly (already at ~94%)
- new_perc_breadth: May increase slightly if they learned about new programs
- new_info_spending_more: May decrease if transparency needs are met

RESPONSE FORMAT (valid JSON only):
{
  "new_trust": 0.0 or 0.5 or 1.0,
  "new_impact": 0.0 or 0.5 or 1.0,
  "new_alignment_pct": float between 0.0 and 100.0,
  "new_perc_breadth": integer between 0 and 8,
  "new_info_spending_more": 0 or 1,
  "qualitative_rationale": "brief explanation in donor's voice (2-3 sentences max)"
}"""

def create_personalized_intervention(alignment_pct, tot25_medium):
    """Create intervention text with donor-specific values."""
    # Convert alignment to 0-100 scale if needed
    if alignment_pct <= 1.0:
        alignment_display = alignment_pct * 100
    else:
        alignment_display = alignment_pct
    
    # Calculate breakdown
    amount_education = tot25_medium * 0.35
    amount_health = tot25_medium * 0.28
    amount_water = tot25_medium * 0.22
    amount_overhead = tot25_medium * 0.15
    
    return INTERVENTION_TEMPLATE.format(
        alignment_pct=alignment_display,
        tot25_medium=tot25_medium,
        amount_education=amount_education,
        amount_health=amount_health,
        amount_water=amount_water,
        amount_overhead=amount_overhead
    )

def create_user_prompt(conditioning_prompt, alignment_pct, tot25_medium, current_trust, current_impact):
    """Create the user prompt with personalized intervention."""
    intervention = create_personalized_intervention(alignment_pct, tot25_medium)
    
    # Provide guidance on expected changes
    alignment_display = alignment_pct * 100 if alignment_pct <= 1.0 else alignment_pct
    
    return f"""{conditioning_prompt}

---

INTERVENTION SCENARIO:
{intervention}

---

As this specific donor, respond to the intervention by updating your attitudes and explaining your reaction.

CONTEXT FOR YOUR RESPONSE:
- Your current trust: {current_trust:.1f} (likely to increase when shown concrete proof)
- Your current impact perception: {current_impact:.1f} (likely to increase when shown measurable results)
- Your current alignment: {alignment_display:.0f}% (already high, may stay same or increase slightly)

Output a JSON object with these exact fields:
- new_trust: 0.0, 0.5, or 1.0 (consider: does concrete proof increase your trust?)
- new_impact: 0.0, 0.5, or 1.0 (consider: does measurable impact increase your belief?)
- new_alignment_pct: number between 0-100 (likely stays at {alignment_display:.0f} since already well-aligned)
- new_perc_breadth: integer between 0-8 (may increase if you learned about programs)
- new_info_spending_more: 0 or 1 (may decrease if transparency needs are met)
- qualitative_rationale: brief explanation in your voice (2-3 sentences max)

Output ONLY the JSON object."""

class GroqClient:
    def __init__(self, api_key, model):
        self.api_key = api_key
        self.model = model
        self.base_url = "https://api.groq.com/openai/v1/chat/completions"
    
    async def generate(self, session, system_prompt, user_prompt, temperature, max_tokens):
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        
        payload = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            "temperature": temperature,
            "max_tokens": max_tokens
        }
        
        async with session.post(self.base_url, headers=headers, json=payload) as response:
            response.raise_for_status()
            data = await response.json()
            return data["choices"][0]["message"]["content"]

async def process_single_donor(donor_id, row, client, session, semaphore):
    """Process a single donor with personalized intervention."""
    async with semaphore:
        # Extract donor-specific values
        conditioning_prompt = row['conditioning_prompt']
        alignment_pct = row.get('alignment_pct', 0.94)  # Default to 94% if missing
        tot25_medium = row.get('tot25_medium', 50.0)    # Default to €50 if missing
        current_trust = row.get('idx_trust', 0.0)
        current_impact = row.get('idx_impact', 0.0)
        
        user_prompt = create_user_prompt(
            conditioning_prompt, 
            alignment_pct, 
            tot25_medium,
            current_trust,
            current_impact
        )
        
        for attempt in range(CONFIG["max_retries"]):
            try:
                raw_response = await asyncio.wait_for(
                    client.generate(session, SYSTEM_PROMPT, user_prompt, CONFIG["temperature"], CONFIG["max_tokens"]),
                    timeout=CONFIG["timeout_seconds"]
                )
                
                # Clean response (handle <think> tags and markdown)
                clean_response = raw_response.strip()
                
                # Remove <think> reasoning blocks
                if "<think>" in clean_response:
                    clean_response = clean_response.split("</think>")[-1].strip()
                
                # Remove markdown code blocks
                if "```json" in clean_response:
                    clean_response = clean_response.split("```json")[1].split("```")[0].strip()
                elif "```" in clean_response:
                    clean_response = clean_response.split("```")[1].split("```")[0].strip()
                
                # Extract JSON object if there's extra text
                if not clean_response.startswith("{"):
                    start = clean_response.find("{")
                    end = clean_response.rfind("}") + 1
                    if start != -1 and end != 0:
                        clean_response = clean_response[start:end]
                    else:
                        raise ValueError(f"No JSON object found")
                
                response_data = json.loads(clean_response)
                
                required_fields = ["new_trust", "new_impact", "new_alignment_pct", "new_perc_breadth", "new_info_spending_more", "qualitative_rationale"]
                
                if not all(f in response_data for f in required_fields):
                    raise ValueError("Missing required fields")
                
                # Add metadata
                response_data["donor_id"] = donor_id
                response_data["success"] = True
                response_data["attempts"] = attempt + 1
                
                # Calculate deltas
                response_data["delta_trust"] = response_data["new_trust"] - current_trust
                response_data["delta_impact"] = response_data["new_impact"] - current_impact
                
                print(f"✓ Donor {donor_id:3d} | Trust: {current_trust:.1f}→{response_data['new_trust']:.1f} | Impact: {current_impact:.1f}→{response_data['new_impact']:.1f}")
                return response_data
                
            except Exception as e:
                if attempt < CONFIG["max_retries"] - 1:
                    print(f"⚠ Donor {donor_id:3d} error (attempt {attempt + 1}): {str(e)[:50]}")
                    await asyncio.sleep(2 ** attempt)
                else:
                    return {
                        "donor_id": donor_id,
                        "new_trust": None,
                        "new_impact": None,
                        "new_alignment_pct": None,
                        "new_perc_breadth": None,
                        "new_info_spending_more": None,
                        "qualitative_rationale": f"FAILED: {str(e)[:100]}",
                        "success": False,
                        "attempts": CONFIG["max_retries"],
                        "delta_trust": None,
                        "delta_impact": None
                    }

async def run_simulation(df):
    client = GroqClient(CONFIG["groq_api_key"], CONFIG["groq_model"])
    semaphore = asyncio.Semaphore(CONFIG["concurrent_limit"])
    
    print(f"\n{'='*70}")
    print(f"🚀 UNICEF PROOF OF IMPACT INTERVENTION SIMULATOR")
    print(f"{'='*70}")
    print(f"Model:      {CONFIG['groq_model']}")
    print(f"Donors:     {len(df)}")
    print(f"Strategy:   Proof of Impact (High Alignment)")
    print(f"Parallel:   {CONFIG['concurrent_limit']} requests")
    print(f"{'='*70}\n")
    
    start_time = time.time()
    
    timeout = aiohttp.ClientTimeout(total=CONFIG["timeout_seconds"])
    async with aiohttp.ClientSession(timeout=timeout) as session:
        tasks = []
        for idx, row in df.iterrows():
            task = process_single_donor(
                donor_id=idx, 
                row=row,
                client=client, 
                session=session, 
                semaphore=semaphore
            )
            tasks.append(task)
        
        results = await asyncio.gather(*tasks)
    
    elapsed_time = time.time() - start_time
    results_df = pd.DataFrame(results)
    
    print(f"\n{'='*70}")
    print(f"✅ COMPLETE!")
    print(f"{'='*70}")
    print(f"Success:    {results_df['success'].sum()}/{len(results_df)}")
    print(f"Time:       {elapsed_time/60:.1f} minutes")
    print(f"{'='*70}\n")
    
    return results_df

def main():
    if not CONFIG["groq_api_key"]:
        print("❌ ERROR: Please set the GROQ_API_KEY environment variable!")
        return
    
    print("📂 Loading donor data...")
    try:
        df = pd.read_csv(CONFIG['input_file'])
        print(f"✓ Loaded {len(df)} donors\n")
    except FileNotFoundError:
        print(f"❌ ERROR: '{CONFIG['input_file']}' not found!")
        print("Run the prompt generation script first!")
        return
    
    if 'conditioning_prompt' not in df.columns:
        print("❌ ERROR: 'conditioning_prompt' column not found!")
        return
    
    # Verify required columns
    required_cols = ['alignment_pct', 'tot25_medium', 'idx_trust', 'idx_impact']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"⚠️  WARNING: Missing columns: {missing_cols}")
        print("Will use default values for missing data.\n")
    
    print("🔄 Starting simulation...\n")
    results_df = asyncio.run(run_simulation(df))
    
    print("💾 Saving results...")
    final_df = df.merge(results_df, left_index=True, right_on='donor_id', how='left')
    final_df.to_csv(CONFIG['output_file'], index=False)
    
    print(f"✓ Saved to '{CONFIG['output_file']}'\n")
    
    if results_df['success'].sum() > 0:
        successful = results_df[results_df['success'] == True]
        
        print("📊 RESULTS SUMMARY:")
        print(f"\n  PRE-INTERVENTION BASELINE (from input data):")
        if 'idx_trust' in df.columns:
            print(f"    Avg Trust:     {df['idx_trust'].mean():.2f}")
        if 'idx_impact' in df.columns:
            print(f"    Avg Impact:    {df['idx_impact'].mean():.2f}")
        
        print(f"\n  POST-INTERVENTION (simulated):")
        print(f"    Avg Trust:     {successful['new_trust'].mean():.2f}")
        print(f"    Avg Impact:    {successful['new_impact'].mean():.2f}")
        print(f"    Avg Alignment: {successful['new_alignment_pct'].mean():.1f}%")
        print(f"    Avg Breadth:   {successful['new_perc_breadth'].mean():.1f}/8")
        
        if 'delta_trust' in successful.columns:
            print(f"\n  CHANGES:")
            print(f"    Δ Trust:       {successful['delta_trust'].mean():+.2f}")
            print(f"    Δ Impact:      {successful['delta_impact'].mean():+.2f}")
            print(f"    % Improved:    {100 * (successful['delta_trust'] > 0).mean():.1f}%")
        
        print(f"\n📝 Sample responses:")
        for i in range(min(3, len(successful))):
            print(f'\n  Donor {i+1}: "{successful.iloc[i]["qualitative_rationale"]}"')
    
    print(f"\n{'='*70}")
    print("✅ READY FOR R POST-INTERVENTION SCORING!")
    print(f"{'='*70}")
    print(f"\nUse file: {CONFIG['output_file']}")
    print("Run: src/04_post_intervention_scoring/post_intervention_scoring.R in R\n")

if __name__ == "__main__":
    main()
